# SnapKV LongBench on Qwen3.5-0.8B

Minimal [LongBench](https://arxiv.org/abs/2308.14508) reproduction using this repo's **SnapKV** port ([`modify_qwen.py`](modify_qwen.py)), adapted from SnapKV's [`pred_snap.py`](https://github.com/FasterDecoding/SnapKV/blob/main/experiments/LongBench/pred_snap.py).

**Runs:** Full-KV baseline vs SnapKV-compressed (`max_capacity_prompt=2048`, `window_size=32`, `kernel_size=7`, `maxpool`).

**Hardware:** GPU recommended (A100-class for long contexts). Upload or clone this repo so `modify_qwen.py` is importable.

**Smoke test:** set `SMOKE_TEST = True` (2 samples, `qasper` only) before a full run.

Citations: LongBench (Bai et al., 2023); SnapKV (Li et al., [arXiv:2404.14469](https://arxiv.org/abs/2404.14469)).

In [ ]:
# Optional Colab drive mount
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
!pip install -q -U transformers datasets accelerate rouge jieba fuzzywuzzy python-Levenshtein tqdm pandas matplotlib

In [ ]:
import copy
import json
import os
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoConfig, AutoTokenizer

REPO_ROOT = Path('.').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from modify_qwen import SnapKVQwen3_5ForCausalLM
from longbench_metrics import scorer

CONFIG_DIR = REPO_ROOT / 'longbench_config'
dataset2prompt = json.loads((CONFIG_DIR / 'dataset2prompt.json').read_text(encoding='utf-8'))
dataset2maxlen = json.loads((CONFIG_DIR / 'dataset2maxlen.json').read_text(encoding='utf-8'))

In [ ]:
# --- Configuration ---
MODEL_ID = 'Qwen/Qwen3.5-0.8B'

ALL_DATASETS = [
    'narrativeqa', 'qasper', 'multifieldqa_en', 'hotpotqa', '2wikimqa', 'musique',
    'gov_report', 'qmsum', 'multi_news', 'trec', 'triviaqa', 'samsum',
    'passage_count', 'passage_retrieval_en', 'lcc', 'repobench-p',
]
DEFAULT_DATASETS = ['qasper', 'hotpotqa', 'narrativeqa']

# Smoke test: 2 samples on qasper only. Set False for DEFAULT_DATASETS or all tasks.
SMOKE_TEST = False
RUN_ALL_DATASETS = False
SAMPLE_LIMIT = None  # e.g. 5 for quick dev; None = full test split
USE_LONGBENCH_E = False

if SMOKE_TEST:
    DATASETS = ['qasper']
    SAMPLE_LIMIT = 2
elif RUN_ALL_DATASETS:
    DATASETS = ALL_DATASETS
else:
    DATASETS = DEFAULT_DATASETS

base_cfg = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
text_cfg = getattr(base_cfg, 'text_config', None) or base_cfg
MAX_INPUT_TOKENS = getattr(text_cfg, 'max_position_embeddings', 32768) - 512

BASELINE_BUDGET = dict(
    max_capacity_prompt=MAX_INPUT_TOKENS,
    window_size=32,
    kernel_size=7,
    pooling='maxpool',
)
SNAPKV_BUDGET = dict(
    max_capacity_prompt=2048,
    window_size=32,
    kernel_size=7,
    pooling='maxpool',
)

RUNS = {
    'qwen35-0.8b-baseline': BASELINE_BUDGET,
    'qwen35-0.8b-snapkv-c2048-w32': SNAPKV_BUDGET,
}

NO_CHAT_WRAP = {'trec', 'triviaqa', 'samsum', 'lsht', 'lcc', 'repobench-p'}

print('Datasets:', DATASETS)
print('MAX_INPUT_TOKENS:', MAX_INPUT_TOKENS)

In [ ]:
def seed_everything(seed: int = 42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def patch_snapkv_budget(cfg, **kwargs):
    c = copy.deepcopy(cfg)
    for obj in filter(None, (c, getattr(c, 'text_config', None))):
        for key, value in kwargs.items():
            setattr(obj, key, value)
    return c


def apply_snapkv_budget_to_model(model, budget: dict):
    inner_cfg = model.model.language_model.config
    for layer_idx, layer in enumerate(model.model.language_model.layers):
        if inner_cfg.layer_types[layer_idx] == 'linear_attention':
            continue
        cache = layer.self_attn.snap_kv_cache
        cache.window_size = budget['window_size']
        cache.max_capacity_prompt = budget['max_capacity_prompt']
        cache.kernel_size = budget['kernel_size']
        cache.pooling = budget['pooling']


def load_snapkv_model(budget: dict):
    cfg = patch_snapkv_budget(base_cfg, **budget)
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    device_kw = {'device_map': 'cuda:0'} if torch.cuda.is_available() else {'device_map': 'cpu'}
    model = SnapKVQwen3_5ForCausalLM.from_pretrained(
        MODEL_ID,
        config=cfg,
        torch_dtype=dtype,
        trust_remote_code=True,
        attn_implementation='eager',
        **device_kw,
    ).eval()
    apply_snapkv_budget_to_model(model, budget)
    return model


def build_prompt(sample: dict, dataset: str) -> str:
    return dataset2prompt[dataset].format(**sample)


def truncate_middle(tokenizer, prompt: str, max_length: int) -> str:
    tokenized = tokenizer(prompt, truncation=False, return_tensors='pt').input_ids[0]
    if len(tokenized) <= max_length:
        return prompt
    half = int(max_length / 2)
    return tokenizer.decode(tokenized[:half], skip_special_tokens=True) + tokenizer.decode(
        tokenized[-half:], skip_special_tokens=True
    )


def tokenize_prompt(tokenizer, prompt: str, dataset: str):
    if dataset in NO_CHAT_WRAP:
        return tokenizer(prompt, truncation=False, return_tensors='pt')
    if not getattr(tokenizer, 'chat_template', None):
        return tokenizer(prompt, truncation=False, return_tensors='pt')
    text = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
    return tokenizer(text, truncation=False, return_tensors='pt')

In [ ]:
@torch.inference_mode()
def predict_dataset(model, tokenizer, dataset: str, out_path: Path, max_input_tokens: int):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists():
        out_path.unlink()

    hf_name = f'{dataset}_e' if USE_LONGBENCH_E else dataset
    data = load_dataset('THUDM/LongBench', hf_name, split='test')
    if SAMPLE_LIMIT is not None:
        data = data.select(range(min(SAMPLE_LIMIT, len(data))))

    max_gen = dataset2maxlen[dataset]
    device = model.device

    for sample in tqdm(data, desc=f'predict:{dataset}'):
        prompt = build_prompt(sample, dataset)
        prompt = truncate_middle(tokenizer, prompt, max_input_tokens)
        batch = tokenize_prompt(tokenizer, prompt, dataset)
        batch = {k: v.to(device) for k, v in batch.items()}
        context_length = batch['input_ids'].shape[-1]

        gen_kwargs = dict(
            max_new_tokens=max_gen,
            num_beams=1,
            do_sample=False,
            temperature=1.0,
            min_length=context_length + 1,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
        )
        if dataset == 'samsum':
            gen_kwargs['eos_token_id'] = [
                tokenizer.eos_token_id,
                tokenizer.encode('\n', add_special_tokens=False)[-1],
            ]

        output = model.generate(**batch, **gen_kwargs)[0]
        pred = tokenizer.decode(output[context_length:], skip_special_tokens=True)

        record = {
            'pred': pred,
            'answers': sample['answers'],
            'all_classes': sample.get('all_classes', None),
            'length': sample.get('length', None),
        }
        with out_path.open('a', encoding='utf-8') as f:
            f.write(json.dumps(record, ensure_ascii=False) + '\n')


def evaluate_predictions(pred_dir: Path) -> dict:
    scores = {}
    for path in sorted(pred_dir.glob('*.jsonl')):
        dataset = path.stem
        predictions, answers, all_classes = [], [], None
        with path.open('r', encoding='utf-8') as f:
            for line in f:
                row = json.loads(line)
                predictions.append(row['pred'])
                answers.append(row['answers'])
                all_classes = row['all_classes']
        scores[dataset] = scorer(dataset, predictions, answers, all_classes)
    return scores


def run_longbench_suite(run_name: str, budget: dict):
    seed_everything(42)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    tokenizer.padding_side = 'left'
    model = load_snapkv_model(budget)

    pred_root = REPO_ROOT / 'pred' / run_name
    for dataset in DATASETS:
        predict_dataset(
            model,
            tokenizer,
            dataset,
            pred_root / f'{dataset}.jsonl',
            MAX_INPUT_TOKENS,
        )

    scores = evaluate_predictions(pred_root)
    result_dir = REPO_ROOT / 'results' / run_name
    result_dir.mkdir(parents=True, exist_ok=True)
    with (result_dir / 'result.json').open('w', encoding='utf-8') as f:
        json.dump(scores, f, ensure_ascii=False, indent=2)
    print(json.dumps(scores, indent=2))
    return scores

In [ ]:
baseline_scores = run_longbench_suite('qwen35-0.8b-baseline', BASELINE_BUDGET)

In [ ]:
snapkv_scores = run_longbench_suite('qwen35-0.8b-snapkv-c2048-w32', SNAPKV_BUDGET)

In [ ]:
rows = []
for dataset in DATASETS:
    rows.append({
        'dataset': dataset,
        'baseline': baseline_scores.get(dataset),
        'snapkv': snapkv_scores.get(dataset),
        'delta': (snapkv_scores.get(dataset) or 0) - (baseline_scores.get(dataset) or 0),
    })

comparison = pd.DataFrame(rows)
comparison_path = REPO_ROOT / 'results' / 'comparison.csv'
comparison_path.parent.mkdir(parents=True, exist_ok=True)
comparison.to_csv(comparison_path, index=False)
display(comparison)

try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(10, 4))
    x = np.arange(len(comparison))
    width = 0.35
    ax.bar(x - width / 2, comparison['baseline'], width, label='baseline')
    ax.bar(x + width / 2, comparison['snapkv'], width, label='snapkv')
    ax.set_xticks(x, comparison['dataset'], rotation=45, ha='right')
    ax.set_ylabel('score')
    ax.set_title('LongBench: baseline vs SnapKV (Qwen3.5-0.8B)')
    ax.legend()
    fig.tight_layout()
    plt.show()
except Exception as exc:
    print('Plot skipped:', exc)